# Building Dimensions table for the constructors silver tables

### Getting the batch id as input parameter

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00.Common/01.Environment-config

In [0]:
%run "../00.Common/04.Helper_Notebook_Gold"

In [0]:
target_name = f"{catalog_name}.{gold_schema}.dim_constructors"
constructors_table = f"{catalog_name}.{silver_schema}.constructors"
nationality_table = f"{catalog_name}.{gold_schema}.ref_nationality_region"

###  Reading the silver tables

In [0]:
#import the sql function and filter the df with the batch id
from pyspark.sql import functions as F
constructors_df = spark.table(constructors_table).filter(F.col("batch_id")==v_batch_id)
nationality_df = spark.table(nationality_table)

### Join the nationality dataframe with the constructors dataframe and renaming the region column name

In [0]:
dim_constructors_df = (
    constructors_df.join(
        nationality_df, constructors_df.nationality == nationality_df.nationality, "left"
        ).select(constructors_df.constructor_id, 
                 constructors_df.constructor_name,
                 constructors_df.nationality, 
                 nationality_df.region.alias("nationality_region"))
)

In [0]:
display(dim_constructors_df)

In [0]:
dim_constructors_df.columns

### Write the table into gold schema

In [0]:
write_to_gold(
    input_df = dim_constructors_df,
    target_table = target_name,
    merge_condition = "t.constructor_id=s.constructor_id",
    columns_to_update = ['constructor_id', 'constructor_name', 'nationality', 'nationality_region']
)

In [0]:
display(spark.table(target_name))